# Meteoritos: de un fichero CSV a un proyecto de análisis
### Computación Científica I

Este cuaderno tiene dos partes:

| Parte | Qué haces | Cómo se trabaja |
|---|---|---|
| **Parte 1. Demo guiada con retos** | Recorremos juntos las funcionalidades obligatorias del proyecto (lectura, estadísticas, filtros, informe y gráficos) con el fichero real. Tras cada bloque hay un **reto** corto. | En clase, individual o por parejas |
| **Parte 2. Vuestro proyecto** | Convertís lo aprendido en `proyecto.py`, tomáis decisiones de diseño como equipo y elegís una **ruta de exploración**. | En equipo (3 o 4 personas) |

**Reglas del juego (del enunciado):**
* Solo Python estándar más las librerías indicadas: `matplotlib`, `reverse_geocoder` y `pycountry_convert`. Por eso aquí **no usamos pandas**: todo se hace con `csv`, listas, diccionarios y tuplas.
* Podéis usar Copilot en Visual Studio Code. **No** se permite generar el código con chats de IA (tipo ChatGPT).
* Cada función del proyecto debe documentar su algoritmo: **entradas, salidas y proceso**. En este cuaderno verás ese formato en todas las funciones.

No hace falta descargar nada a mano: la primera celda de código descarga `meteoritos.csv` desde la web de la NASA si no está ya en la carpeta del cuaderno.

# Parte 1. Demo guiada con retos

## 0. Los datos: de dónde vienen

El fichero que usamos es el conjunto **Meteorite Landings**, publicado en el **Portal de Datos Abiertos de la NASA**. Los datos los recopila **The Meteoritical Society**, la asociación científica internacional que mantiene el registro oficial de meteoritos (el *Meteoritical Bulletin*). Contiene unos 45.000 meteoritos conocidos, tanto los que se vieron caer como los que se encontraron después.

**Antes de seguir, abre la página de documentación del conjunto de datos:**

**https://data.nasa.gov/dataset/meteorite-landings**

Fíjate en:
* qué organización produce los datos y cuándo se actualizaron por última vez;
* qué formatos de descarga se ofrecen (CSV, JSON, XML...);
* qué columnas tiene el fichero y qué significa cada una (compáralas con la tabla del enunciado).

Leer la documentación **antes** de programar es una buena práctica: te dirá, por ejemplo, en qué unidades está la masa o cómo se codifican las fechas. Y lo que la documentación *no* dice (valores ausentes, coordenadas raras, años imposibles) lo descubrirás explorando, como haremos en los retos.

La celda siguiente descarga el CSV con `urllib.request`, que forma parte de la biblioteca estándar de Python. Solo descarga si el fichero no existe ya, así que puedes ejecutarla tantas veces como quieras. Después trabajaremos siempre con el **fichero local**, porque la lectura con operaciones de E/S es lo que pide el enunciado.

In [ ]:
import os
import urllib.request

URL_DATOS = "https://data.nasa.gov/docs/legacy/meteorite_landings/Meteorite_Landings.csv"
RUTA_DATOS = "meteoritos.csv"

def descargar_si_falta(url, ruta):
    """Descarga un fichero desde una URL solo si no existe ya en disco.

    Entradas: url (str) de origen, ruta (str) del fichero local.
    Salidas:  True si el fichero está disponible al terminar, False si no se pudo descargar.
    Proceso:  si la ruta ya existe no hace nada; si no, lo descarga con urllib.request
              y captura los errores de red o de escritura.
    """
    if os.path.exists(ruta):
        print(f"'{ruta}' ya existe; no se descarga de nuevo.")
        return True
    try:
        print("Descargando datos desde", url, "...")
        urllib.request.urlretrieve(url, ruta)
        print(f"Guardado como '{ruta}' ({os.path.getsize(ruta) / 1e6:.1f} MB)")
        return True
    except OSError as error:   # incluye errores de red (URLError) y de disco
        print("No se pudo descargar el fichero:", error)
        print("Descárgalo a mano desde la página de documentación y guárdalo como", ruta)
        return False


descargar_si_falta(URL_DATOS, RUTA_DATOS)

## 1. Lectura del fichero CSV

Antes de escribir código, piensa en la **estructura de datos**. Cada fila del CSV es un meteorito con campos de distinto tipo. Una opción natural es:

* **Un meteorito = un diccionario**: las claves son los nombres de los campos, así el código se lee solo (`m["masa"]` en lugar de `m[4]`).
* **Todos los meteoritos = una lista de diccionarios**: conserva el orden del fichero y se recorre con un `for`.

| Campo CSV | Clave en nuestro diccionario | Tipo Python | Si falta el dato |
|---|---|---|---|
| `name` | `nombre` | `str` | (siempre existe) |
| `id` | `id` | `int` | se descarta la fila |
| `nametype` | `tipo_nombre` | `str` | |
| `recclass` | `clase` | `str` | |
| `mass (g)` | `masa` | `float` | `None` |
| `fall` | `estado` | `str` (`"Fell"` / `"Found"`) | |
| `year` | `anio` | `int` | `None` |
| `reclat`, `reclong` | `lat`, `lon` | `float` | `None` |
| `GeoLocation` | (no se guarda) | | es redundante con `lat` y `lon` |

**¿Por qué `None` y no `0`?** Porque `0` es un valor válido que contaminaría las estadísticas (una masa de 0 g bajaría la media; una coordenada (0, 0) es un punto real del océano Atlántico). `None` significa "no lo sabemos" y nos obliga a decidir qué hacer con él.

Primero, dos funciones auxiliares de conversión. Fíjate en que **nunca lanzan una excepción**: si el texto no se puede convertir, devuelven `None`.

In [ ]:
import csv

def a_float(texto):
    """Convierte un texto en float.

    Entradas: texto (str), por ejemplo "720" o "".
    Salidas:  float, o None si el texto está vacío o no es un número.
    Proceso:  intenta float(texto); si falla, captura la excepción y devuelve None.
    """
    try:
        return float(texto)
    except (ValueError, TypeError):
        return None


def a_anio(texto):
    """Extrae el año de un texto.

    Entradas: texto (str). Acepta "1952" y también "01/01/1952 12:00:00 AM",
              que es el formato del fichero original de la NASA.
    Salidas:  int con el año, o None si no se puede interpretar.
    Proceso:  si hay barras '/', se queda con los 4 primeros caracteres del tercer trozo;
              después intenta convertir a entero.
    """
    if texto is None:
        return None
    texto = texto.strip()
    if "/" in texto:
        partes = texto.split("/")
        if len(partes) < 3:
            return None
        texto = partes[2][:4]
    try:
        return int(float(texto))
    except ValueError:
        return None


print(a_float("720"), a_float(""), a_anio("1952"), a_anio("01/01/1880 12:00:00 AM"), a_anio("??"))

Ahora la función de carga. Gestiona los dos errores que pide el enunciado:

1. **Fichero no encontrado**: `FileNotFoundError`.
2. **Formato incorrecto**: comprobamos que la cabecera tenga las columnas que necesitamos. Además detectamos si el separador es `,` o `;` (Excel en español suele guardar con `;`). Si una fila concreta está corrupta, la contamos y seguimos: una fila mala no debería tirar todo el programa.

In [ ]:
COLUMNAS_NECESARIAS = ["name", "id", "nametype", "recclass", "mass (g)", "fall", "year", "reclat", "reclong"]

def cargar_meteoritos(ruta):
    """Lee el fichero CSV de meteoritos.

    Entradas: ruta (str) del fichero CSV.
    Salidas:  tupla (meteoritos, descartadas):
              meteoritos  -> lista de diccionarios, uno por meteorito
              descartadas -> int, filas que no se pudieron interpretar
              Si el fichero no existe o su formato es incorrecto devuelve ([], 0).
    Proceso:  1. Abre el fichero (captura FileNotFoundError).
              2. Detecta el separador mirando la primera línea.
              3. Comprueba que la cabecera contiene las columnas necesarias.
              4. Convierte cada fila en un diccionario con los tipos adecuados.
    """
    meteoritos = []
    descartadas = 0
    try:
        with open(ruta, encoding="utf-8-sig") as f:
            primera_linea = f.readline()
            separador = ";" if primera_linea.count(";") > primera_linea.count(",") else ","
            f.seek(0)  # volvemos al principio del fichero
            lector = csv.DictReader(f, delimiter=separador)

            cabecera = lector.fieldnames or []
            faltan = [c for c in COLUMNAS_NECESARIAS if c not in cabecera]
            if faltan:
                print("Formato incorrecto. Faltan las columnas:", faltan)
                return [], 0

            for fila in lector:
                try:
                    meteorito = {
                        "nombre": fila["name"].strip(),
                        "id": int(fila["id"]),
                        "tipo_nombre": fila["nametype"],
                        "clase": fila["recclass"],
                        "masa": a_float(fila["mass (g)"]),
                        "estado": fila["fall"],
                        "anio": a_anio(fila["year"]),
                        "lat": a_float(fila["reclat"]),
                        "lon": a_float(fila["reclong"]),
                    }
                    meteoritos.append(meteorito)
                except (ValueError, TypeError, AttributeError):
                    descartadas += 1
    except FileNotFoundError:
        print(f"No se encuentra el fichero '{ruta}'. ¿Está en la misma carpeta que el cuaderno?")
        return [], 0
    return meteoritos, descartadas


meteoritos, descartadas = cargar_meteoritos(RUTA_DATOS)
print(f"Cargados: {len(meteoritos)}   Filas descartadas: {descartadas}")
print("Primer meteorito:", meteoritos[0] if meteoritos else "(ninguno)")

In [ ]:
# Comprobamos que la gestión de errores funciona con un fichero que no existe
prueba, _ = cargar_meteoritos("no_existe.csv")
print("Meteoritos cargados del fichero inexistente:", len(prueba))

### Reto 1. ¿Qué datos faltan?
Cuenta cuántos meteoritos **no tienen** masa, cuántos no tienen año y cuántos no tienen coordenadas. Después, cuenta cuántos tienen coordenadas **exactamente (0, 0)** y cuántos tienen **masa 0**.

*Pregunta para pensar:* ¿crees que hay tantos meteoritos caídos justo en el punto (0, 0) del Atlántico? ¿Qué significa realmente ese valor y cómo debería tratarlo tu programa?

In [ ]:
# TU CÓDIGO AQUÍ

## 2. Estadísticas básicas

### 2a, 2b y 2c. Total, más pesado, más ligero y media

Podríamos usar `max()` con una función clave, pero es más instructivo escribir el **algoritmo de recorrido**, que es exactamente lo que se pide documentar:

> Recorrer la lista una sola vez, guardando "el mejor candidato hasta ahora" para el máximo y el mínimo, y acumulando la suma y la cantidad para la media.

Una sola pasada: el coste es proporcional al número de meteoritos.

In [ ]:
def estadisticas_masa(meteoritos):
    """Calcula estadísticas de masa.

    Entradas: meteoritos (lista de diccionarios).
    Salidas:  diccionario con las claves 'total', 'con_masa', 'mas_pesado', 'mas_ligero', 'media'.
              'mas_pesado' y 'mas_ligero' son diccionarios de meteorito (o None si no hay masas).
    Proceso:  una pasada por la lista ignorando las masas None; se actualizan máximo, mínimo,
              suma y contador; al final media = suma / contador.
    """
    mas_pesado = None
    mas_ligero = None
    suma = 0.0
    contador = 0
    for m in meteoritos:
        masa = m["masa"]
        if masa is None:
            continue
        if mas_pesado is None or masa > mas_pesado["masa"]:
            mas_pesado = m
        if mas_ligero is None or masa < mas_ligero["masa"]:
            mas_ligero = m
        suma += masa
        contador += 1
    media = suma / contador if contador > 0 else None
    return {"total": len(meteoritos), "con_masa": contador,
            "mas_pesado": mas_pesado, "mas_ligero": mas_ligero, "media": media}


est = estadisticas_masa(meteoritos)
print(f"Número total de meteoritos: {est['total']}")
print(f"Más pesado: {est['mas_pesado']['nombre']} ({est['mas_pesado']['masa']:,.0f} g)")
print(f"Más ligero: {est['mas_ligero']['nombre']} ({est['mas_ligero']['masa']} g)")
print(f"Media de masas: {est['media']:,.1f} g  (calculada sobre {est['con_masa']} meteoritos con masa)")

### Reto 2. ¿Es la media un buen resumen?
1. Calcula la **mediana** de las masas (pista: ordena una lista con las masas válidas con `sorted()` y toma el elemento central; cuidado si la cantidad es par).
2. Compara media y mediana. ¿Por qué son tan diferentes?
3. Muestra los **10 meteoritos más pesados** con su nombre, masa en kg y año.

In [ ]:
# TU CÓDIGO AQUÍ

### 2d. Meteoritos por franjas temporales, por estado y combinados

Las tres tablas usan el mismo patrón, el **diccionario contador**:

```python
conteo = {}
for elemento in datos:
    clave = ...                              # ¿por qué agrupo?
    conteo[clave] = conteo.get(clave, 0) + 1
```

Lo único que cambia es la **clave**:
* por franja: un texto como `"1900-1949"`
* por estado: `"Fell"` o `"Found"`
* combinado: una **tupla** `("1900-1949", "Fell")`. Las tuplas pueden ser claves de diccionario porque son inmutables; las listas no.

El ancho de la franja es una **decisión de diseño**: lo dejamos como parámetro.

In [ ]:
def franja_de(anio, ancho=50):
    """Devuelve la etiqueta de la franja temporal de un año.

    Entradas: anio (int), ancho (int) de la franja en años.
    Salidas:  str como "1900-1949".
    Proceso:  inicio = (anio // ancho) * ancho; fin = inicio + ancho - 1.
    """
    inicio = (anio // ancho) * ancho
    return f"{inicio}-{inicio + ancho - 1}"


def contar_por(meteoritos, funcion_clave):
    """Cuenta meteoritos agrupando por la clave que devuelve funcion_clave.

    Entradas: meteoritos (lista), funcion_clave (función que recibe un meteorito y devuelve
              una clave, o None para ignorarlo).
    Salidas:  diccionario {clave: número de meteoritos}.
    Proceso:  patrón diccionario contador.
    """
    conteo = {}
    for m in meteoritos:
        clave = funcion_clave(m)
        if clave is not None:
            conteo[clave] = conteo.get(clave, 0) + 1
    return conteo


ANCHO_FRANJA = 50

def clave_franja(m):
    return franja_de(m["anio"], ANCHO_FRANJA) if m["anio"] is not None else None

def clave_estado(m):
    return m["estado"]

def clave_franja_estado(m):
    if m["anio"] is None:
        return None
    return (franja_de(m["anio"], ANCHO_FRANJA), m["estado"])


por_franja = contar_por(meteoritos, clave_franja)
por_estado = contar_por(meteoritos, clave_estado)
por_ambas = contar_por(meteoritos, clave_franja_estado)

print("Por estado:", por_estado)
print()
print(f"{'Franja':<12}{'Fell':>8}{'Found':>8}{'Total':>8}")
for franja in sorted(por_franja):
    fell = por_ambas.get((franja, "Fell"), 0)
    found = por_ambas.get((franja, "Found"), 0)
    print(f"{franja:<12}{fell:>8}{found:>8}{por_franja[franja]:>8}")

### Reto 3. Elige tus franjas
1. Cambia `ANCHO_FRANJA` a 100 y a 25 y vuelve a ejecutar la celda anterior. ¿Qué ancho cuenta mejor la historia? ¿Hay franjas con tan pocos datos que no dicen nada?
2. Calcula, para cada franja, el **porcentaje de meteoritos observados al caer (`Fell`)**. ¿Cuándo empieza a dominar `Found`? ¿Se te ocurre por qué? (Pista: busca "Antarctic Search for Meteorites".)
3. Revisa los años mínimo y máximo. ¿Hay algún año imposible (por ejemplo, en el futuro)?

In [ ]:
# TU CÓDIGO AQUÍ

### 2e. Porcentaje de meteoritos por continente

El enunciado define cada continente como una **caja** de latitudes y longitudes. Guardamos la tabla en un diccionario cuyo valor es una tupla `(lat_min, lat_max, lon_min, lon_max)`: es un dato constante, así que una tupla (inmutable) es el tipo adecuado.

Antes de programar, **mira las cajas en un mapa** (la celda siguiente). Vas a descubrir dos problemas que tu equipo tendrá que resolver.

In [ ]:
CONTINENTES = {
    "América del Norte": (7.0, 83.0, -172.0, -10.0),
    "América del Sur":   (-55.0, 12.0, -81.0, -35.0),
    "Europa":            (35.0, 71.0, -25.0, 60.0),
    "África":            (-35.0, 37.0, -18.0, 52.0),
    "Asia":              (-1.0, 81.0, 26.0, 180.0),
    "Oceanía":           (-47.0, 10.0, 110.0, 180.0),
}

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Paleta categórica (orden fijo) y colores de texto
COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
COLOR_FELL, COLOR_FOUND = "#2a78d6", "#eb6834"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b",
                     "xtick.color": "#52514e", "ytick.color": "#52514e", "font.size": 10})

con_coord = [m for m in meteoritos
             if m["lat"] is not None and m["lon"] is not None and not (m["lat"] == 0 and m["lon"] == 0)]

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.scatter([m["lon"] for m in con_coord], [m["lat"] for m in con_coord], s=2, color="#52514e", alpha=0.4)
for (nombre, (la0, la1, lo0, lo1)), color in zip(CONTINENTES.items(), COLORES):
    ax.add_patch(Rectangle((lo0, la0), lo1 - lo0, la1 - la0, fill=True, alpha=0.12, color=color))
    ax.add_patch(Rectangle((lo0, la0), lo1 - lo0, la1 - la0, fill=False, lw=2, edgecolor=color))
    ax.text(lo0 + 2, la1 - 6, nombre, color="#0b0b0b", fontsize=9, weight="bold")
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
ax.set_title("Meteoritos con coordenadas y cajas de continentes del enunciado", loc="left")
plt.show()

**Los dos problemas que se ven en el mapa:**

1. **Las cajas se solapan.** Por ejemplo, el sur de España está a la vez en la caja de Europa y en la de África; Turquía en Europa y Asia; el norte de Sudamérica en las dos Américas. Hay que decidir una **regla de desempate**.
2. **Hay zonas sin caja.** Los océanos, y sobre todo la **Antártida**, que no aparece en la tabla. Allí se han encontrado una enorme cantidad de meteoritos (el hielo los concentra y en la nieve se ven fácilmente).

La versión de la demo usa la regla más sencilla: **el primer continente del diccionario que contiene el punto**, y si ninguno lo contiene, `"Sin región"`. No es la única regla posible.

In [ ]:
def dentro_de(lat, lon, caja):
    """Indica si (lat, lon) está dentro de una caja (lat_min, lat_max, lon_min, lon_max)."""
    lat_min, lat_max, lon_min, lon_max = caja
    return lat_min <= lat <= lat_max and lon_min <= lon <= lon_max


def continente_de(lat, lon):
    """Asigna un continente a unas coordenadas.

    Entradas: lat, lon (float o None).
    Salidas:  str con el continente, "Sin coordenadas" o "Sin región".
    Proceso:  recorre CONTINENTES en orden y devuelve el primero que contiene el punto
              (regla de desempate: orden del diccionario). (0, 0) se trata como desconocido.
    """
    if lat is None or lon is None or (lat == 0 and lon == 0):
        return "Sin coordenadas"
    for nombre, caja in CONTINENTES.items():
        if dentro_de(lat, lon, caja):
            return nombre
    return "Sin región"


def porcentajes(conteo):
    """Convierte un diccionario de conteos en otro de porcentajes sobre el total."""
    total = sum(conteo.values())
    return {clave: 100 * valor / total for clave, valor in conteo.items()}


def clave_continente(m):
    return continente_de(m["lat"], m["lon"])

por_continente = contar_por(meteoritos, clave_continente)
pct = porcentajes(por_continente)
for nombre in sorted(pct, key=pct.get, reverse=True):
    print(f"{nombre:<20}{por_continente[nombre]:>7}{pct[nombre]:>8.1f}%")

### Reto 4. ¿Cuánto importan los solapes?
1. Escribe una función `continentes_de(lat, lon)` (en plural) que devuelva una **lista** con *todos* los continentes cuya caja contiene el punto.
2. Cuenta cuántos meteoritos caen en 0, 1, 2 o más cajas.
3. Cuenta cuántos meteoritos tienen latitud menor que -60 (Antártida). ¿Qué porcentaje del total representan?
4. Debate: ¿el porcentaje "por continente" debería calcularse sobre todos los meteoritos o solo sobre los que tienen coordenadas? Escribe tu decisión y tu argumento en una celda Markdown.

In [ ]:
# TU CÓDIGO AQUÍ

## 3. Filtrado y búsqueda

El usuario quiere recuperar parte de los datos por **año**, por **rango de años** o por **nombre**. Todas las funciones de filtro siguen el mismo esquema: reciben la lista completa y **devuelven una lista nueva** con los que cumplen la condición. No modifican la original, así que se pueden **encadenar**.

In [ ]:
def filtrar_por_anios(meteoritos, desde, hasta=None):
    """Filtra por año o por rango de años.

    Entradas: meteoritos (lista), desde (int), hasta (int o None).
              Si hasta es None se filtra por el año exacto 'desde'.
    Salidas:  lista nueva con los meteoritos cuyo año está en [desde, hasta].
    Proceso:  recorrido con condición; los años None se descartan.
    """
    if hasta is None:
        hasta = desde
    resultado = []
    for m in meteoritos:
        if m["anio"] is not None and desde <= m["anio"] <= hasta:
            resultado.append(m)
    return resultado


def buscar_por_nombre(meteoritos, texto):
    """Busca meteoritos cuyo nombre contiene un texto, sin distinguir mayúsculas.

    Entradas: meteoritos (lista), texto (str).
    Salidas:  lista nueva con las coincidencias.
    Proceso:  compara texto.lower() con nombre.lower() usando el operador 'in'.
    """
    texto = texto.lower()
    return [m for m in meteoritos if texto in m["nombre"].lower()]


def mostrar(lista, maximo=10):
    """Imprime una lista de meteoritos como tabla (como mucho 'maximo' filas)."""
    print(f"{'Nombre':<28}{'Año':>6}{'Masa (g)':>14}  {'Estado':<6} Clase")
    for m in lista[:maximo]:
        masa = f"{m['masa']:,.1f}" if m["masa"] is not None else "?"
        print(f"{m['nombre']:<28}{str(m['anio']):>6}{masa:>14}  {m['estado']:<6} {m['clase']}")
    if len(lista) > maximo:
        print(f"... y {len(lista) - maximo} más")


print("Meteoritos de 1950:")
mostrar(filtrar_por_anios(meteoritos, 1950), 5)
print("\nNombre contiene 'madrid':")
mostrar(buscar_por_nombre(meteoritos, "madrid"), 5)
print("\nEncadenando: nombre contiene 'allende' y años 1960-1980:")
mostrar(filtrar_por_anios(buscar_por_nombre(meteoritos, "allende"), 1960, 1980), 5)

### Reto 5. Un filtro nuevo
Escribe `filtrar_por_estado(meteoritos, estado)` y úsala, encadenada con las anteriores, para responder: **¿cuántos meteoritos se vieron caer (`Fell`) entre 1900 y 1999?** ¿Cuál fue el más pesado de ellos? (Reutiliza `estadisticas_masa`.)

*Extra:* escribe `filtrar_por_continente(meteoritos, continente)`.

In [ ]:
# TU CÓDIGO AQUÍ

## 4. Salida en fichero: el informe

La idea clave de diseño: **las funciones de cálculo no imprimen, devuelven datos**. Otra función construye las líneas de texto del informe y una tercera las escribe en disco. Así la misma información sirve para la pantalla, para el fichero y para los gráficos.

In [ ]:
def lineas_informe(meteoritos, titulo):
    """Construye el texto de un informe.

    Entradas: meteoritos (lista, puede ser el resultado de un filtro), titulo (str).
    Salidas:  lista de str, una por línea.
    Proceso:  llama a las funciones de estadísticas y da formato a sus resultados.
    """
    lineas = [titulo, "=" * len(titulo), ""]
    e = estadisticas_masa(meteoritos)
    lineas.append(f"Número de meteoritos: {e['total']}")
    if e["media"] is not None:
        lineas.append(f"Más pesado: {e['mas_pesado']['nombre']} ({e['mas_pesado']['masa']:,.0f} g)")
        lineas.append(f"Más ligero: {e['mas_ligero']['nombre']} ({e['mas_ligero']['masa']} g)")
        lineas.append(f"Media de masas: {e['media']:,.1f} g")
    lineas.append("")
    lineas.append("Por estado:")
    for estado, n in sorted(contar_por(meteoritos, clave_estado).items()):
        lineas.append(f"  {estado:<8}{n:>7}")
    lineas.append("")
    lineas.append("Por continente:")
    conteo = contar_por(meteoritos, clave_continente)
    pct = porcentajes(conteo)
    for nombre in sorted(pct, key=pct.get, reverse=True):
        lineas.append(f"  {nombre:<20}{conteo[nombre]:>7}{pct[nombre]:>8.1f}%")
    return lineas


def escribir_informe(ruta, lineas):
    """Escribe una lista de líneas en un fichero de texto.

    Entradas: ruta (str), lineas (lista de str).
    Salidas:  True si se escribió correctamente, False si hubo un error de E/S.
    """
    try:
        with open(ruta, "w", encoding="utf-8") as f:
            for linea in lineas:
                f.write(linea + "\n")
        return True
    except OSError as error:
        print("No se pudo escribir el informe:", error)
        return False


lineas = lineas_informe(meteoritos, "Informe general de meteoritos")
if escribir_informe("informe_general.txt", lineas):
    print("Informe guardado en informe_general.txt\n")
print("\n".join(lineas))

### Reto 6. Informe de una consulta
Genera el fichero `informe_siglo_XX.txt` con el informe de los meteoritos **de 1900 a 1999**. Añade al informe una sección nueva con la tabla de franjas de 10 años de ese siglo.

In [ ]:
# TU CÓDIGO AQUÍ

## 5. Gráficos con matplotlib (funcionalidad opcional 1)

Tres gráficos, cada uno elegido por la **pregunta** que responde:

| Pregunta | Gráfico |
|---|---|
| ¿Cómo se reparten las masas? | Histograma del **logaritmo** de la masa (las masas van de gramos a toneladas: en escala lineal todo se aplasta en la primera barra) |
| ¿Cómo cambia Fell/Found con el tiempo? | Barras agrupadas por franja |
| ¿Dónde caen? | Dispersión latitud/longitud coloreada por estado |

In [ ]:
import math

log_masas = [math.log10(m["masa"]) for m in meteoritos if m["masa"] is not None and m["masa"] > 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(log_masas, bins=40, color=COLOR_FELL, edgecolor="white", linewidth=1)
ax.set_xticks(range(-1, 9))
ax.set_xticklabels(["0,1 g", "1 g", "10 g", "100 g", "1 kg", "10 kg", "100 kg", "1 t", "10 t", "100 t"])
ax.set_xlabel("Masa (escala logarítmica)")
ax.set_ylabel("Número de meteoritos")
ax.set_title("Distribución de masas", loc="left")
plt.show()

In [ ]:
franjas = sorted(por_franja)
fell = [por_ambas.get((f, "Fell"), 0) for f in franjas]
found = [por_ambas.get((f, "Found"), 0) for f in franjas]
x = range(len(franjas))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([i - 0.2 for i in x], fell, width=0.38, color=COLOR_FELL, label="Fell (observado al caer)")
ax.bar([i + 0.2 for i in x], found, width=0.38, color=COLOR_FOUND, label="Found (hallado)")
ax.set_xticks(list(x))
ax.set_xticklabels(franjas, rotation=45, ha="right")
ax.set_ylabel("Número de meteoritos")
ax.set_title(f"Meteoritos por franja de {ANCHO_FRANJA} años y estado", loc="left")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for estado, color, tam in [("Found", COLOR_FOUND, 2), ("Fell", COLOR_FELL, 8)]:
    puntos = [m for m in con_coord if m["estado"] == estado]
    ax.scatter([m["lon"] for m in puntos], [m["lat"] for m in puntos], s=tam, color=color,
               alpha=0.6, label=f"{estado} ({len(puntos)})")
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
ax.set_title("¿Dónde están los meteoritos? Observados frente a hallados", loc="left")
ax.legend(frameon=False, loc="lower left", markerscale=3)
plt.show()

### Reto 7. Tu gráfico
Elige **una** de estas preguntas y responde con un gráfico (título, ejes con nombre y leyenda si hay más de una serie):

* ¿Qué porcentaje de meteoritos hay en cada continente? (barras horizontales ordenadas)
* ¿Las masas de los observados (`Fell`) son distintas de las de los hallados (`Found`)? (dos histogramas del logaritmo de la masa)
* ¿Cuáles son las 10 clases (`clase`) más frecuentes?

Escribe debajo, en una celda Markdown, **una frase** con la conclusión que se ve en tu gráfico.

In [ ]:
# TU CÓDIGO AQUÍ

# Parte 2. Vuestro proyecto

En la Parte 1 has visto **una** forma de resolver cada funcionalidad. Ahora vuestro equipo construye `proyecto.py` y **toma decisiones propias**. No hay una única respuesta correcta: se valora que cada decisión esté **justificada** y documentada en la memoria y en la presentación.

## 2.1 Del cuaderno al programa: diseño arquitectónico

El enunciado pide que **cada función sea un módulo** y que se represente el **intercambio de datos con el programa principal**. Un diagrama posible:

```
                         ┌──────────────────────────┐
                         │     programa principal   │
                         │  (menú en bucle: input)  │
                         └────────────┬─────────────┘
      ruta (str)                      │
   ┌──────────────────────────────────┼──────────────────────────────────────┐
   ▼                                  ▼                                      ▼
cargar_meteoritos               estadisticas_*                       filtrar_* / buscar_*
  -> (lista de dict, int)         lista -> dict                        lista, criterio -> lista
                                      │                                      │
                                      ▼                                      ▼
                               lineas_informe  -> lista de str  -> escribir_informe -> fichero .txt
                                      │
                                      ▼
                               graficar_* (opcional)  -> ventana / fichero .png
```

Regla de oro: **la lista de meteoritos se carga una vez** en el programa principal y se pasa como argumento. Ninguna función usa variables globales salvo las constantes (como `CONTINENTES`).

El esqueleto de abajo es un punto de partida para `proyecto.py`. Copiadlo a un fichero `.py`, sustituid los `pass` y ejecutadlo desde la terminal (`python proyecto.py`): **el menú con `input()` está pensado para la terminal**, no para el cuaderno.

### Esqueleto de `proyecto.py`

```python
# proyecto.py  -  Computación Científica I  -  Equipo: ____________
# Autores: ______, ______, ______, ______

import csv

# ---------- Constantes ----------
RUTA_DATOS = "meteoritos.csv"
CONTINENTES = { ... }          # tabla del enunciado


# ---------- 1. Lectura ----------
def cargar_meteoritos(ruta):
    '''Entradas: ...  Salidas: ...  Proceso: ...'''
    pass


# ---------- 2. Estadísticas ----------
def estadisticas_masa(meteoritos):
    pass

def contar_por(meteoritos, funcion_clave):
    pass

def porcentaje_por_continente(meteoritos):
    pass


# ---------- 3. Filtros ----------
def filtrar_por_anios(meteoritos, desde, hasta=None):
    pass

def buscar_por_nombre(meteoritos, texto):
    pass


# ---------- 4. Informe ----------
def lineas_informe(meteoritos, titulo):
    pass

def escribir_informe(ruta, lineas):
    pass


# ---------- Programa principal ----------
def menu():
    print("1. Estadísticas generales")
    print("2. Filtrar por año o rango de años")
    print("3. Buscar por nombre")
    print("4. Guardar informe de la última consulta")
    print("0. Salir")
    return input("Opción: ")


def main():
    meteoritos, descartadas = cargar_meteoritos(RUTA_DATOS)
    if not meteoritos:
        return
    seleccion = meteoritos          # resultado de la última consulta
    opcion = menu()
    while opcion != "0":
        if opcion == "1":
            ...
        elif opcion == "2":
            ...
        opcion = menu()


if __name__ == "__main__":
    main()
```

## 2.2 Decisiones de diseño del equipo

Rellenad esta tabla **juntos** en la primera reunión y apuntad la decisión en el fichero de seguimiento. Cada fila es material directo para la sección "Descripción de los datos" de la memoria y para la presentación.

| # | Decisión | Opciones posibles | Nuestra decisión | Por qué |
|---|---|---|---|---|
| 1 | Estructura principal | lista de diccionarios / diccionario indexado por `id` / lista de tuplas | | |
| 2 | Datos ausentes | `None` y se ignoran / se descarta la fila / valor por defecto | | |
| 3 | Coordenadas (0, 0) | dato válido / desconocido | | |
| 4 | Ancho de las franjas temporales | 10 / 25 / 50 / 100 años / franjas irregulares (antes de 1800, 1800-1949, ...) | | |
| 5 | Años imposibles | se descartan / se avisa al usuario | | |
| 6 | Solape de cajas de continentes | primero en el orden / caja más pequeña / se cuenta en todos / categoría "frontera" | | |
| 7 | Puntos sin caja (océano, Antártida) | "Sin región" / añadir Antártida y océanos como regiones propias | | |
| 8 | Base de los porcentajes | todos los meteoritos / solo los que tienen coordenadas | | |
| 9 | Búsqueda por nombre | exacta / contiene / sin distinguir mayúsculas ni tildes | | |
| 10 | Formato del informe | texto plano / tabla alineada / CSV | | |
| 11 | Funcionalidad opcional | ruta A, B, C, D o E (ver 2.3) | | |

Celda de configuración: si vuestro código lee estas decisiones de **constantes**, podéis cambiarlas sin tocar las funciones (y enseñar en la presentación cómo cambia el resultado).

In [ ]:
# ---- Configuración del equipo (editad estos valores) ----
ANCHO_FRANJA = 50               # decisión 4
CERO_ES_DESCONOCIDO = True      # decisión 3
REGLA_SOLAPE = "primero"        # decisión 6: "primero", "mas_pequena", "todos"
ANIO_MAXIMO_VALIDO = 2026       # decisión 5

## 2.3 Elegid una ruta de exploración

Cada ruta parte de una **pregunta**. Las rutas A a D usan la funcionalidad opcional del enunciado; la E es libre. Para la ruta elegida, el entregable mínimo es: **una pregunta, al menos dos gráficos, una tabla en el informe y una conclusión de un párrafo** que responda a la pregunta.

### Ruta A. ¿Quién mira al cielo? Observados frente a hallados
*Pregunta guía:* ¿El número de meteoritos registrados dice algo sobre el cielo o sobre las personas que los buscan?
* Proporción `Fell`/`Found` por franja temporal y por continente.
* ¿Desde qué década domina `Found`? Relacionadlo con las campañas de búsqueda en la Antártida y en desiertos (Sáhara, Omán).
* ¿Dónde se *observan* caer más meteoritos? ¿Coincide con zonas muy pobladas?
* Gráficos sugeridos: barras apiladas por franja en porcentaje; dispersión de solo `Fell`.

### Ruta B. Geografía de verdad: cajas frente a fronteras
*Pregunta guía:* ¿Cuánto se equivoca el método de las cajas?
* Usad `reverse_geocoder` y `pycountry_convert` (celda de ejemplo más abajo) para asignar **país y continente reales**.
* Construid una **tabla de confusión**: filas = continente según cajas, columnas = continente según fronteras. ¿Dónde están los errores?
* Cuidado: `reverse_geocoder` devuelve la **ciudad más cercana**, aunque el punto esté en mitad del océano o en la Antártida. ¿Cómo lo detectáis?
* Gráfico sugerido: mapa con los puntos mal clasificados resaltados.

### Ruta C. ¿De qué están hechos?
*Pregunta guía:* ¿Qué tipos de meteoritos son más frecuentes y cuáles son más pesados?
* La columna `recclass` tiene cientos de valores. Diseñad una función que la agrupe en **familias** (por ejemplo: condritas ordinarias H, L y LL; condritas carbonáceas C...; hierros `Iron...`; acondritas; pallasitas). La regla de agrupación es vuestra decisión.
* Masa media y mediana por familia. ¿Los hierros son más pesados? ¿Por qué tendría sentido?
* ¿Hay familias que se *encuentran* mucho más de lo que se *observan* caer?

### Ruta D. Más cerca de casa: Europa y España
*Pregunta guía:* ¿Qué país europeo tiene más meteoritos y qué provincias españolas?
* Con `reverse_geocoder`, el campo `cc` da el país y `admin1`/`admin2` la región y la provincia.
* Ranking de países europeos; meteoritos en España por comunidad o provincia.
* Ficha de los meteoritos españoles (nombre, año, masa, estado). ¿Alguno cayó cerca de vuestra ciudad?

### Ruta E. Ruta libre
Proponed vuestra propia pregunta. Debe usar al menos dos campos del fichero combinados y responderse con gráficos y tablas. Validadla con el profesorado antes de empezar.

### Ejemplo de uso de `reverse_geocoder` y `pycountry_convert` (rutas B y D)

Instalación (una vez, en la terminal): `pip install reverse_geocoder pycountry_convert`

Dos detalles prácticos:
* Pasad **todas las coordenadas a la vez** en una lista: una llamada por meteorito es muy lenta.
* En cuadernos y en Windows usad `mode=1` (un solo proceso) para evitar errores de multiproceso.

In [ ]:
try:
    import reverse_geocoder as rg
    import pycountry_convert as pc
    LIBRERIAS_GEO = True
except ImportError:
    LIBRERIAS_GEO = False
    print("Faltan librerías. Instálalas con: pip install reverse_geocoder pycountry_convert")


NOMBRES_CONTINENTE = {"EU": "Europa", "AF": "África", "AS": "Asia", "NA": "América del Norte",
                      "SA": "América del Sur", "OC": "Oceanía", "AN": "Antártida"}

def continente_real(codigo_pais):
    """Convierte un código de país ISO alfa-2 en nombre de continente (o 'Desconocido')."""
    if codigo_pais == "AQ":
        return "Antártida"
    try:
        return NOMBRES_CONTINENTE[pc.country_alpha2_to_continent_code(codigo_pais)]
    except KeyError:
        return "Desconocido"


if LIBRERIAS_GEO:
    # Ejemplo con Madrid
    print(rg.search([(40.4168, -3.7038)], mode=1))

    # Todos los meteoritos con coordenadas de una sola vez
    coordenadas = [(m["lat"], m["lon"]) for m in con_coord]
    lugares = rg.search(coordenadas, mode=1)
    for m, lugar in zip(con_coord, lugares):
        m["pais"] = lugar["cc"]
        m["region"] = lugar["admin1"]
        m["continente_real"] = continente_real(lugar["cc"])

    print("\nPrimeros 5 meteoritos con país asignado:")
    for m in con_coord[:5]:
        print(f"{m['nombre']:<25} {m['pais']}  {m['region']:<25} {m['continente_real']}")

**Para pensar antes de fiaros del resultado:** un meteorito encontrado en la Antártida, ¿qué "ciudad más cercana" os devuelve `reverse_geocoder`? Comparad `continente_real` con `continente_de` (cajas) para los puntos con latitud menor que -60.

## 2.4 Trabajo en equipo

### Plantilla de acta para el fichero de seguimiento
Copiad esta plantilla al inicio de **cada** reunión (un par de párrafos, nada extenso):

```
## Reunión n.º __  -  Fecha: __/__/____
Asistentes: ...
Objetivos de hoy: ...
Tareas de la reunión anterior: [hecha / en curso / pendiente] tarea - responsable
Desarrollo: (2-4 frases: qué se hizo, qué problemas surgieron, qué se decidió)
Próxima reunión: tareas y responsables
  - ... -> ...
  - ... -> ...
```

### Reparto sugerido de roles (rotad si queréis)
| Rol | Responsabilidad |
|---|---|
| Coordinación | Convoca reuniones, mantiene el acta al día |
| Datos | Lectura del CSV, tipos, validación y casos raros |
| Estadísticas e informe | Funciones de cálculo y salida en fichero |
| Exploración y gráficos | Ruta elegida y matplotlib |

Todas las personas deben poder explicar **todo** el código: habrá un examen de autoría y todos participan en la presentación.

## 2.5 Lista de comprobación antes de entregar

**Código (`proyecto.py`)**
- [ ] Se ejecuta de principio a fin con `python proyecto.py` sin errores.
- [ ] Solo usa Python estándar y las librerías permitidas.
- [ ] Gestiona fichero no encontrado y formato incorrecto.
- [ ] Cada función tiene docstring con **entradas, salidas y proceso**.
- [ ] Ninguna función usa variables globales salvo constantes.
- [ ] Nombres de variables y funciones descriptivos y comentarios útiles.

**Funcionalidad obligatoria**
- [ ] Total, más pesado, más ligero y media de masas.
- [ ] Conteos por franja, por estado y combinados.
- [ ] Porcentaje por continente, con la regla de solapes documentada.
- [ ] Filtros por año, rango de años y nombre.
- [ ] Informe en fichero de texto.

**Documentación**
- [ ] Memoria: introducción, diseño arquitectónico, datos, detalles del código, manual de usuario, conclusiones, uso de IA.
- [ ] Fichero de seguimiento con actas de cada sesión hasta la presentación.
- [ ] Presentación: problema, arquitectura con intercambio de datos, datos y tipos elegidos, algoritmos, código de interés, reparto del trabajo.